In [1]:
## Importing X_train, X_test, y_train, y_test
import pandas as pd

X_train = pd.read_csv("../dataset/X_train.csv")
X_test = pd.read_csv("../dataset/X_test.csv")
y_train = pd.read_csv("../dataset/y_train.csv").to_numpy().ravel()
y_test = pd.read_csv("../dataset/y_test.csv").to_numpy().ravel()

## Scaling

In [2]:
from sklearn.preprocessing import StandardScaler

numeric_col = [
    "Purchase_Quantity",
    "Shelf_Life_Days",
    "Storage_Temperature",
    "Daily_Demand",
]

scaler = StandardScaler()

X_train[numeric_col] = scaler.fit_transform(X_train[numeric_col])
X_test[numeric_col] = scaler.transform(X_test[numeric_col])

In [3]:
print(X_train.shape)
print(X_test.shape)

(6400, 23)
(1600, 23)


In [4]:
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score,
)

In [5]:
result = []

In [6]:
models = {
    "LinearRegression": LinearRegression(),
    "KNN": KNeighborsRegressor(n_neighbors=5),
    "SVM": SVR(C=100),
    "DT": DecisionTreeRegressor(),
}
n = X_test.shape[0]
p = X_test.shape[1]

In [7]:
for name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)

    mse = mean_squared_error(y_true=y_test, y_pred=pred)
    mae = mean_absolute_error(y_true=y_test, y_pred=pred)
    r2 = r2_score(y_true=y_test, y_pred=pred)

    adjusted_r2 = 1 - ((1 - r2) * (n - 1) / (n - p - 1))

    result.append(
        {
            "model": name,
            "mae": round(mae, 4),
            "mse": round(mse, 4),
            "r2": round(r2, 4),
            "adjusted_r2": round(adjusted_r2, 4),
        }
    )

In [8]:
result

[{'model': 'LinearRegression',
  'mae': 4.1881,
  'mse': 59.9775,
  'r2': 0.8037,
  'adjusted_r2': 0.8009},
 {'model': 'KNN',
  'mae': 3.5909,
  'mse': 46.9564,
  'r2': 0.8463,
  'adjusted_r2': 0.8441},
 {'model': 'SVM',
  'mae': 1.54,
  'mse': 34.3994,
  'r2': 0.8874,
  'adjusted_r2': 0.8858},
 {'model': 'DT',
  'mae': 1.7581,
  'mse': 30.5181,
  'r2': 0.9001,
  'adjusted_r2': 0.8987}]

## Fine Tuning

In [9]:
from sklearn.model_selection import GridSearchCV

param_grid_svr = [
    {
        "kernel": ["rbf"],
        "C": [0.1, 1, 10, 100],
        "gamma": ["scale", "auto", 0.01, 0.1],
        "epsilon": [0.1, 0.5, 1],
    },
    {"kernel": ["linear"], "C": [0.1, 1, 10, 100], "epsilon": [0.1, 0.5, 1]},
]

In [10]:
svr_grid = GridSearchCV(
    estimator=SVR(), param_grid=param_grid_svr, scoring="r2", cv=5, n_jobs=-1, verbose=1
)

In [ ]:
svr_grid.fit(X_train, y_train)

Fitting 5 folds for each of 60 candidates, totalling 300 fits


In [ ]:
print("Best Parameters:", svr_grid.best_params_)
print("Best CV R²:", svr_grid.best_score_)

svr_model = svr_grid.best_estimator_

Best Parameters: {'C': 100, 'epsilon': 1, 'gamma': 0.01, 'kernel': 'rbf'}
Best CV R²: 0.9394322824642881


In [ ]:
from sklearn.metrics import r2_score

y_train_pred = svr_model.predict(X_train)
y_test_pred = svr_model.predict(X_test)

print("Training R²:", r2_score(y_train, y_train_pred))
print("Testing R²:", r2_score(y_test, y_test_pred))
print("Training MAE:", mean_absolute_error(y_train, y_train_pred))
print("Testing MAE:", mean_absolute_error(y_test, y_test_pred))
print("Training MSE:", mean_squared_error(y_train, y_train_pred))
print("Testing MSE:", mean_squared_error(y_test, y_test_pred))

Training R²: 0.9539415624637817
Testing R²: 0.9530993429630833
Training MAE: 1.5084162919884636
Testing MAE: 1.5996285469172546
Training MSE: 12.059666211260172
Testing MSE: 14.331794301251044


In [ ]:
import joblib

joblib.dump(svr_model, filename="../models/SVR_Food_Waste.pkl")
joblib.dump(scaler, filename="../models/scaler.pkl")
joblib.dump(X_train.columns.to_list(), filename="../models/columns.pkl")

['../models/columns.pkl']